# Fase 2 — Data Understanding

**Objetivo:** explorar os dados sem alterá-los ainda, formando 
hipóteses sobre qualidade e relação com o target.

## 1. Objetivo da fase 

Explorar os dados sem alterá-los ainda (trabalho da Fase 3 — Data Preparation), formando hipóteses sobre qualidade dos dados e relação de cada variável com o target. O objetivo aqui não é "limpar", é entender e documentar problemas, cada problema que encontrarmos vira uma decisão justificada na próxima fase.

**Pré-requisito:** cs-training.csv em data/raw/ (veja o README na raiz do projeto para instruções de download).  

## 2. Dicionário de Variáveis

### Target (Variável a Prever)

- **`SeriousDlqin2yrs`:** Pessoa teve atraso de 90+ dias nos próximos 
  2 anos (0=Não, 1=Sim)

### Features (Variáveis Explicativas)

| Variável | Descrição | Tipo |
|----------|-----------|------|
| `RevolvingUtilizationOfUnsecuredLines` | % de utilização do crédito disponível | Contínua (0-∞) |
| `age` | Idade em anos | Inteiro |
| `NumberOfTime30-59DaysPastDueNotWorse` | Quantas vezes atrasou 30-59 dias | Inteiro |
| `DebtRatio` | Razão dívida/renda mensal | Contínua |
| `MonthlyIncome` | Renda mensal | Contínua |
| `NumberOfOpenCreditLinesAndLoans` | Quantidade de linhas de crédito abertas | Inteiro |
| `NumberOfTimes90DaysLate` | Quantas vezes atrasou 90+ dias | Inteiro |
| `NumberRealEstateLoansOrLines` | Empréstimos imobiliários | Inteiro |
| `NumberOfTime60-89DaysPastDueNotWorse` | Quantas vezes atrasou 60-89 dias | Inteiro |
| `NumberOfDependents` | Dependentes na família | Inteiro |

---

### Target

**`SeriousDlqin2yrs`**
- **Descrição:** Pessoa teve atraso de 90+ dias nos próximos 2 anos
- **Valores:** 0 = Não, 1 = Sim
- **Tipo:** Binária (classificação)
- **Expectativa:** Classes desbalanceadas (~6-7% de 1's)

---

### Features - Grupo 1: Comportamento de Crédito Passado

**`NumberOfTimes90DaysLate`**
- **Descrição:** Quantas vezes atrasou 90+ dias
- **Tipo:** Inteiro (≥0)
- **Expectativa:** Forte preditor! Quem já atrasou 90+ dias tem mais chance de atrasar de novo
- **Hipótese de negócio:** Deve ter correlação ALTA com o target

**`NumberOfTime60-89DaysPastDueNotWorse`**
- **Descrição:** Quantas vezes atrasou 60-89 dias
- **Tipo:** Inteiro (≥0)
- **Expectativa:** Preditor importante, mas menos forte que 90+ dias

**`NumberOfTime30-59DaysPastDueNotWorse`**
- **Descrição:** Quantas vezes atrasou 30-59 dias
- **Tipo:** Inteiro (≥0)
- **Expectativa:** Sinal de risco, mas mais "leve"

---

### Features - Grupo 2: Situação Financeira Atual

**`RevolvingUtilizationOfUnsecuredLines`**
- **Descrição:** % de utilização do crédito disponível (revolving credit)
- **Tipo:** Contínua (0-∞, mas esperado 0-1)
- **Fórmula:** saldo_utilizado / limite_disponível
- **Expectativa:** Valores >1 podem indicar over-limit ou erro de dados
- **Hipótese:** Alta utilização (próximo de 1) = maior risco

**`DebtRatio`**
- **Descrição:** Razão dívida/renda mensal
- **Tipo:** Contínua (0-∞)
- **Fórmula:** (pagamentos_mensais + despesas) / renda_bruta_mensal
- **Expectativa:** Valores muito altos podem ser outliers
- **Hipótese:** Alto debt ratio = maior risco

**`MonthlyIncome`**
- **Descrição:** Renda mensal
- **Tipo:** Contínua (>0)
- **Expectativa:** Pode ter valores faltantes, outliers extremos
- **Hipótese:** Renda mais alta = menor risco (inversamente proporcional)

---

### Features - Grupo 3: Perfil de Crédito

**`NumberOfOpenCreditLinesAndLoans`**
- **Descrição:** Quantidade de linhas de crédito abertas (cartões + empréstimos)
- **Tipo:** Inteiro (≥0)
- **Expectativa:** Relação não-linear? (ter muito pouco OU muito pode ser ruim)

**`NumberRealEstateLoansOrLines`**
- **Descrição:** Empréstimos imobiliários (hipotecas, home equity)
- **Tipo:** Inteiro (≥0)
- **Hipótese:** Ter imóvel pode ser sinal de estabilidade (menor risco)

---

### Features - Grupo 4: Demográficas

**`age`**
- **Descrição:** Idade em anos
- **Tipo:** Inteiro (>0)
- **Expectativa:** Muito jovens (<18) ou muito velhos (>100) podem ser erros
- **Hipótese:** Relação não-linear (jovens demais = risco; muito velhos também?)

**`NumberOfDependents`**
- **Descrição:** Dependentes na família (excluindo o próprio)
- **Tipo:** Inteiro (≥0)
- **Expectativa:** Pode ter valores faltantes
- **Hipótese:** Mais dependentes = mais despesas = maior risco?

## 3. Roteiro de EDA (Exploratory Data Analysis)

Este roteiro segue uma progressão lógica:   

**verificação inicial → qualidade dos dados → distribuições → relações → síntese**  

Cada etapa responde uma pergunta específica e gera saídas concretas que informam decisões na Fase 3 (Data Preparation).


### Etapas

| Etapa | Pergunta que você está respondendo | Ferramentas | Saída esperada |
|-------|-----------------------------------|-------------|----------------|
| **0.Carregamento e Verificação** | O dataset foi carregado corretamente com tamanho e formato esperados? | `shape`, `head()`, `tail()`, `dtypes`, `info()` | Confirmação: 150.000 linhas × 11 colunas, tipos corretos |
| **1a. Estatísticas Descritivas** | Quais são os ranges, médias e distribuições básicas de cada variável? | `describe()`, `value_counts()` | Tabela anotada com min/max/mean/std de cada coluna |
| **1b. Valores Impossíveis** | Existem valores logicamente impossíveis ou suspeitos? | Filtros condicionais (ex: `df[df['age'] < 0]`) | Lista documentada de inconsistências (idade negativa, utilização > 10, etc.) |
| **2a. Dados Faltantes - Quantidade** | Quantos dados estão faltando e em quais colunas? | `isna().sum()`, `isna().mean()`, visualização (missingno) | % de missing por variável + visualização |
| **2b. Dados Faltantes - Padrão** | Os dados faltam de forma aleatória (MAR) ou há um padrão sistemático (MNAR)? | Análise condicional (ex: missing de renda por idade), tabelas cruzadas | Hipótese: MAR vs. MNAR + evidências |
| **3. Registros Duplicados** | Existem linhas completamente duplicadas no dataset? | `duplicated().sum()`, inspeção de duplicados | Contagem de duplicados + decisão (manter/remover/investigar) |
| **4. Distribuição do Target** | Qual o grau de desbalanceamento da variável target? | `value_counts()`, `value_counts(normalize=True)`, gráfico de barras | ~6.6% de 1's (inadimplentes) → confirma necessidade de AUC/Recall em vez de acurácia |
| **5. Análise Univariada** | Como cada feature se distribui isoladamente? Há outliers? | `histplot`, `boxplot`, `violinplot`, estatísticas de quartis | Histogramas + boxplots anotados, lista de outliers extremos por variável |
| **6a. Análise Bivariada - Visual** | Como as distribuições das features variam entre bons e maus pagadores? | `boxplot` por target, `violinplot` por target, distribuições separadas | Padrões visuais: quais variáveis "separam" bem as classes? |
| **6b. Análise Bivariada - Quantitativa** | Qual a força da relação de cada feature com o target? | Correlação point-biserial, taxas de evento por faixa/bin | Ranking de features por poder preditivo (preparação para feature selection) |
| **7. Correlação Entre Features** | Quais variáveis explicativas são redundantes ou altamente correlacionadas? | `corr()`, heatmap triangular, lista de pares | Identificação de pares com \|r\| > 0.7 → decisão sobre qual manter na Fase 3 |
| **8. Síntese e Hipóteses para Fase 3** | O que aprendemos e quais decisões de preparação antecipamos? | — | Lista estruturada: problemas encontrados → ações planejadas |






 ### **0. Carregamento e Verificação**

In [ ]:
# Libraries 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import os 

In [ ]:
from pathlib import Path

def project_root():
    here = Path.cwd().resolve()
    for pasta in [here, *here.parents]:
        if (pasta / "data" / "raw" / "cs-training.csv").exists():
            return pasta
    raise FileNotFoundError("cs-training.csv não está em data/raw/")

ROOT = project_root()
print("ROOT:", ROOT)

df = pd.read_csv(ROOT / "data/raw/cs-training.csv")
print(df.shape)

## 1a. Estatísticas descritivas 
Visão geral das estatísticas do Dataset. 

- Visão Geral  
df.shape          # Dimensão (linhas, colunas)  
df.info()         # Tipos de dados + memória + não-nulos  
df.head()         # Primeiras 5 linhas  
df.tail()         # Últimas 5 linhas  

- Estatísticas Numéricas  
df.describe()     # Estatísticas das variáveis numéricas  
df.describe(include='all')  # Inclui categóricas também  

- Tipos e Estrutura  
df.dtypes         # Tipo de cada coluna  
df.columns.tolist()  # Lista de colunas  

- Valores Únicos e Distribuição  
df.nunique()      # Quantos valores únicos em cada coluna  
df['coluna'].value_counts()  # Frequência de cada valor  

- Verificações Específicas  
df.duplicated().sum()  # Quantas linhas duplicadas  
df.isna().sum()   # Quantos valores faltantes por coluna  
df.isna().mean() * 100  # % de valores faltantes  

In [ ]:
# Visão geral do Dataset 
# Dimensão (linhas, colunas)
df.shape

# Carregado corretamente 

In [ ]:
# Tipos de dados + memória + não-nulos
df.info()

**Carregamento:** 150.000 linhas × 12 colunas (10 features + target `SeriousDlqin2yrs` + `Unnamed: 0` = índice, não feature).

Missing explícito só em `MonthlyIncome` e `NumberOfDependents` (quantificar na 2a; padrão na 2b). Target sem NaN.
 

In [ ]:
# Primeiras cinco linhas
df.head()

In [ ]:
# Últimas 5 linhas
df.tail()

In [ ]:
# Estatística variáveis numéricas
df.describe()

### Observações 

- RevolvingUtilizationOfUnsecuredLines   

Média de 604% está sendo puxada por outliers, numero parece irreal. Temos que investigar.
Desvio padrão de 249%, muito extremo para uma medida que é mensurada de 0-1.   
Mediana (50%) parece estar correta com 15,41% de utilização de crédito.   
Max é um extremo outlier.   


In [ ]:
s = df['RevolvingUtilizationOfUnsecuredLines']  # não altere df

print("n (não-nulo):", s.notna().sum())
print("% missing:", s.isna().mean() * 100)
print("iguais a 0:", (s == 0).sum())
print(s.describe())
print(s.quantile([0.0, 0.25, 0.5, 0.9, 0.99, 0.999, 1.0]))

Cortes na base (150k): **3.321** com utilização > 1 (~2,2%) e **241** com utilização > 10 (~0,16%).

- `> 1` **não** é automaticamente erro: over-limit pode ser real. O P99 ≈ 1,09 mostra que a maioria desses 3.321 está só um pouco acima de 100%.
- Os **241** com `> 10` (e o max 50.708) **não** são ratio de cartão, são dado quebrado. A média 6,05 vem dessa cauda, não do cliente típico (mediana ≈ 0,15).

**Fase 3:** cap P99 e/ou flag `> 10`; não tratar os dois grupos iguais; não usar a média.

- age   

Média de 52 anos dos clientes, com desvio padrão de 14. Parece estar correto.   
Min de 0, temos que investigar, não tem como uma pessoa com 0 anos tomar crédito.   
Max de 109, parece suspeito, é um outlier extremo.   

In [ ]:
s = df['age']  # não altere df

print("n (não-nulo):", s.notna().sum())
print("% missing:", s.isna().mean() * 100)
print("iguais a 0:", (s == 0).sum())
print(s.describe())
print(s.quantile([0.0, 0.25, 0.5, 0.9, 0.99, 0.999, 1.0]))

- NumberOfTime30-59DaysPastDueNotWorse  

Média de 0.42 com alto desvio padrão, significa presença de outliers.  
Até 75% das pessoas não atrasaram nenhuma vez, mas temos outliers de 98, ou seja, algum cliente teve até 98 atrasos, o que está puxando para um alta variância.   
Na prática é impossivel existir a presença de 98 ciclos de atraso, sendo que o target olha para dois anos, ou seja, seria no máximo 24 meses de atraso. Temos que investigar esse outlier.  


In [ ]:
# Ver a distribuição de valores
s = df['NumberOfTime30-59DaysPastDueNotWorse']  # não altere df

print("n (não-nulo):", s.notna().sum())
print("% missing:", s.isna().mean() * 100)
print("iguais a 0:", (s == 0).sum())
print(s.describe())
print(s.quantile([0.0, 0.25, 0.5, 0.9, 0.99, 0.999, 1.0]))

- DebtRatio  

Metade das pessoas utilizam 36% da renda mensal com a divida e até 75% das pessoas utilizam 87% da renda mensal com a divida.    
Média 353% é inutil, ninguem "deve 353 vezes a renda".   
Claramente outliers para 1% das pessoas no qual a razão da divida pela renda mensal foi superior a 1267. 
Sem MonthlyIncome, o arquivo muitas vezes não divide por renda (denominador ausente ou = 1). O número vira dívida em unidade absoluta (ou lixo), e isso puxa média, P90, P99 e max.  


In [ ]:
s = df['DebtRatio']  # não altere df

print("n (não-nulo):", s.notna().sum())
print("% missing:", s.isna().mean() * 100)
print("iguais a 0:", (s == 0).sum())
print(s.describe())
print(s.quantile([0.0, 0.25, 0.5, 0.9, 0.99, 0.999, 1.0]))

- MonthlyIncome    

Temos cerca de 20% dos dados missing, ou seja, um problema que temos que investigar.   
Média 6.67k e mediana de 5.4k significa que até 50% das pessoas ganham 5.4k ou menos e temos outliers que fazem essa média subir para 6.67k.   
min = 0: investigar se é renda real zero ou missing recodificado.    
max extremo = outlier; conferir percentis 99/99,9 antes de chamar de “erro”.    


In [ ]:
s = df['MonthlyIncome']  # não altere df

print("n (não-nulo):", s.notna().sum())
print("% missing:", s.isna().mean() * 100)
print("iguais a 0:", (s == 0).sum())
print(s.describe())
print(s.quantile([0.0, 0.25, 0.5, 0.9, 0.99, 0.999, 1.0]))


- NumberOfOpenCreditLinesAndLoans  

A pessoa típica tem cerca de 8 linhas abertas. O desvio de 5,14 indica que a carteira não é homogênea: é comum estar alguns pontos acima ou abaixo de 8 (por exemplo na casa de 3–13), sem ser o caos dos contadores de atraso (muitos zeros + 96/98).

In [ ]:
s = df['NumberOfOpenCreditLinesAndLoans']  # não altere df

print("n (não-nulo):", s.notna().sum())
print("% missing:", s.isna().mean() * 100)
print("iguais a 0:", (s == 0).sum())
print(s.describe())
print(s.quantile([0.0, 0.25, 0.5, 0.9, 0.99, 0.999, 1.0]))

- NumberOfTimes90DaysLate  

Mais de 90 % dos clientes nunca atrasaram mais do que 90 dias e apenas 1% dos clientes atrasaram 3 vezes e 0,1% dos clientes atrasaram até 98 vezes. 

In [ ]:
s = df['NumberOfTimes90DaysLate']  # não altere df

print("n (não-nulo):", s.notna().sum())
print("% missing:", s.isna().mean() * 100)
print("iguais a 0:", (s == 0).sum())
print(s.describe())
print(s.quantile([0.0, 0.25, 0.5, 0.9, 0.99, 0.999, 1.0]))

- NumberRealEstateLoansOrLines  

Média das pessoas é 1 empréstimo imobiliário com 75% das pessoas até 2 empréstimos. Apenas 1% das pessoas com 4 ou mais empréstimos imobiliários. 

In [ ]:
s = df['NumberRealEstateLoansOrLines']  # não altere df

print("n (não-nulo):", s.notna().sum())
print("% missing:", s.isna().mean() * 100)
print("iguais a 0:", (s == 0).sum())
print(s.describe())
print(s.quantile([0.0, 0.25, 0.5, 0.9, 0.99, 0.999, 1.0]))

- NumberOfTime60-89DaysPastDueNotWorse  

Até 75% das pessoas nunca atrasaram até 89 dias e apenas 1% atrasaram 2 vezes ou mais. Apenas 0,1% atrasaram 98 vezes.  
Temos que investigar esse outlier, não tem como atrasar 98 ciclos sendo que o target olha para o periodo de dois anos, o maximo seria atrasar 24 meses.  

In [ ]:
s = df['NumberOfTime60-89DaysPastDueNotWorse']  # não altere df

print("n (não-nulo):", s.notna().sum())
print("% missing:", s.isna().mean() * 100)
print("iguais a 0:", (s == 0).sum())
print(s.describe())
print(s.quantile([0.0, 0.25, 0.5, 0.9, 0.99, 0.999, 1.0]))

- NumberOfDependents  

Até 75% não tem dependente na família e 10% da base tem 2 ou mais.  
Há **~3.924 NaN (~2,6%)** — `isna()` não aparece no `describe` da mesma forma; tratar na 2a/2b. Max 20: cauda rara, baixa prioridade.
 


In [ ]:
s = df['NumberOfDependents']  # não altere df

print("n (não-nulo):", s.notna().sum())
print("% missing:", s.isna().mean() * 100)
print("iguais a 0:", (s == 0).sum())
print(s.describe())
print(s.quantile([0.0, 0.25, 0.5, 0.9, 0.99, 0.999, 1.0]))

In [ ]:
# Estatística numéricas de todas variáveis
df.describe(include='all')  


In [ ]:
# Tipo de estrutura de cada coluna
df.dtypes

In [ ]:
# Lista de colunas 
df.columns.tolist()

In [ ]:
# Valores unicos e distribuição 
# Quantos valores únicos em cada coluna
df.nunique()

In [ ]:
# Porcentagem da variável Target
# Base desbalanceada
# 0 = Não 
# 1 = Sim 
df["SeriousDlqin2yrs"].value_counts(normalize=True) * 100

In [ ]:
# Quantas linhas duplicadas
df.duplicated().sum()

# Não temos linhas duplicadas no Dataset

In [ ]:
# Quantos valores faltantes por coluna
df.isna().sum()

In [ ]:
# Porcentagem de valores faltantes
df.isna().mean() * 100 

Temos quase 20% de valores faltantes na coluna "MonthlyIncome" e pouco mais de 2.5% na coluna "NumberOfDependents".   
Veremos como tratar esses dados.  

## **1b. Valores Impossíveis**

Separar o que o modelo pode tratar como numero real do que é extremo. Vai responder perguntas como: "Esse valor pode existir no negócio de crédito ?"

| Tipo | Exemplo neste dataset | O que fazer depois |
|---|---|---|
| **Impossível** | `age == 0` (1 linha); 96/98 "atrasos" | Corrigir/recodificar/remover — não é idade nem 98 eventos |
| **Extremo possível** | `age = 109`; 4 pessoas | Baixa prioridade; às vezes manter |
| **Ambíguo** | `renda = 0`; Target com `utilização > 1` | Decidir com contexto (2b, P99) |

In [ ]:
# --- 1b. Valores impossíveis / suspeitos (não altera df) ---

print("age < 18:", (df['age'] < 18).sum())
print("age == 0:", (df['age'] == 0).sum())
print("age > 100:", (df['age'] > 100).sum())

u = df['RevolvingUtilizationOfUnsecuredLines']
print("utilização % crédito disponível > 1:", (u > 1).sum())
print("utilização % crédito disponível > 10:", (u > 10).sum())

print("DebtRatio > 1:", (df['DebtRatio'] > 1).sum())
print("DebtRatio > 100:", (df['DebtRatio'] > 100).sum())

atrasos = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
]
for c in atrasos:
    print(c, ">= 96:", (df[c] >= 96).sum())

# Os mesmos clientes nas 3 colunas?
mask_98 = (
    (df['NumberOfTime30-59DaysPastDueNotWorse'] >= 96)
    & (df['NumberOfTime60-89DaysPastDueNotWorse'] >= 96)
    & (df['NumberOfTimes90DaysLate'] >= 96)
)
print("linhas com >=96 nas TRÊS colunas de atraso:", mask_98.sum())

print("MonthlyIncome == 0:", (df['MonthlyIncome'] == 0).sum())
print("NumberOfDependents > 10:", (df['NumberOfDependents'] > 10).sum())

In [ ]:
# Achando no dataset quem é o cliente age=0
df.loc[df['age'] == 0, ['age', 'SeriousDlqin2yrs', 'MonthlyIncome']]


Alguem digitou errado essa linha,  idade 0, renda 5.000, target 0 → não é bebê. Apaga ou corrige essa linha. 

In [ ]:
# Para ver as pessoas com mais de 1000% de utilização de crédito disponivel
df.loc[u > 10, ['RevolvingUtilizationOfUnsecuredLines', 'DebtRatio', 'MonthlyIncome']].head(10)

241 pessoas com utilização maior que 10 (1000% do limite).   
O valor da utilização é cortado no P99 do treino. 

#### **Conclusões**

Idade.   
age < 18 = 1 e age == 0 = 1 → a mesma linha. Não há menores na base; há um cadastro com idade 0. Os 13 com age > 100 são extremos possíveis (P99 = 87); prioridade baixa. O loc da idade 0 serve para ver se o resto do cadastro é de adulto (renda/target) — se for, é erro pontual: drop ou imputar no 03.

RevolvingUtilizationOfUnsecuredLines.   
3.321 > 1 (~2%) vs 241 > 10. Dois regimes: over-limit pode ser real; > 10 não é ratio de cartão. O segundo loc (util, DebtRatio, renda) mostra se a cauda vem junto com renda/DebtRatio estranhos. Fase 3: cap P99 (treino) e/ou flag > 10; não tratar os 3.321 iguais aos 241.

DebtRatio.   
35.137 > 1 (~23%) não são 23% de “erro”. Dívida > renda existe. Já 24.380 > 100 (~16%) não são ratio — e o tamanho é da mesma ordem dos 29.731 sem renda. A 2b liga os dois: sem renda o campo vira número enorme. Fase 3: cap separado com flag de renda; não apagar 35k linhas com ratio > 1.

Renda 0.  
 1.634: desemprego ou missing disfarçado. Não é impossível. Nesta fase: manter 0, imputar só NaN.

Dependentes > 10.   
2 linhas. Irrelevante para regra de negócio.

### Resumo para a síntese da Fase 2
| Caso | Tipo | Prioridade Fase 3 |
|------|------|-------------------|
| `age == 0` (1) | Impossível | Alta (simples) |
| 96/98 nas colunas de atraso | Código / impossível como contagem | Alta |
| Utilização > 10 e DebtRatio > 100 | Dado quebrado / não-ratio | Alta |
| Utilização > 1 (sem ser > 10) | Suspeito, às vezes real | Média |
| Renda = 0 | Ambíguo | Média (junto com missing) |
| `age > 100`, dependentes > 10 | Extremo possível | Baixa |



## **2a. Dados Faltantes - Quantidade**

In [ ]:
# Quantos valores faltantes por coluna
df.isna().sum()

In [ ]:
# Dados faltantes em porcentagem
df.isna().mean() * 100 

Missing explícito (`NaN`) em **2 colunas** de 12:

| Variável | N | % da base (150k) |
|----------|---|------------------|
| `MonthlyIncome` | 29.731 | 19,82% |
| `NumberOfDependents` | 3.924 | 2,62% |
| Demais | 0 | 0% |

- **Renda** é o problema de volume: qualquer modelo treinado “dropando NaN” perde ~20% da amostra e **enviesa** se quem não informa renda for diferente (testar na 2b).
- **Dependentes** é imputação simples em volume; a pergunta é se o missing coincide com o de renda.
- Target **sem** missing → rótulo completo (bom para classificação).
- `isna()` **não** captura renda 0 nem códigos 96/98 (tratar como qualidade à parte / 1b).

**Fase 3:** não dropar 29k linhas de renda sem 2b; planejar imputação + **flag** `income_was_missing` (e o análogo de dependentes se o padrão for informativo).

## **2b. Dados Faltantes - Padrão**

O fato de temos dados faltantes muda as outras colunas que estão preenchidas?

In [ ]:
# Flags de missing (True = NaN). Não altera o df.
# Compara valores missing nas duas colunas "MonthlyIncome" e "NumberOfDependents" 

# Alguém que não declara renda também não declara dependentes
inc_na = df['MonthlyIncome'].isna()
dep_na = df['NumberOfDependents'].isna()

# Renda missing e dependentes NÃO missing
print("só renda missing:", (inc_na & ~dep_na).sum())
# Renda NÃO missing e dependentes missing 
print("só dependentes missing:", (~inc_na & dep_na).sum())
# Os dois missing simultaneamente
print("os dois missing:", (inc_na & dep_na).sum())
# Nenhum missing simultaneamente 
print("nenhum missing:", (~inc_na & ~dep_na).sum())

# Taxa de inadimplência (PD) por missing
print("PD (taxa de 1) com renda informada:", df.loc[~inc_na, 'SeriousDlqin2yrs'].mean())
print("PD com renda missing:", df.loc[inc_na, 'SeriousDlqin2yrs'].mean())

print("PD dependentes informados:", df.loc[~dep_na, 'SeriousDlqin2yrs'].mean())
print("PD dependentes missing:", df.loc[dep_na, 'SeriousDlqin2yrs'].mean())


Renda informada: PD = 6,95%.   
Renda missing: PD = 5,61%.  
Dependentes informados: 6,74%.   
Dependentes missing: 4,56%.  
Base: ~6,68%.  

O missing **não é aleatório** em relação ao default, mas o sinal é o **oposto** do “buraco = mais risco”: quem não tem renda/dependentes preenchidos **default menos**.

Implicações Fase 3:
- Não dropar NaN (sairiam os clientes melhores).
- Imputar **e** manter flag (`income_was_missing`, e de dependentes): o vazio ainda distingue grupos.
- Não tratar missing como proxy automático de alto risco.

Próximo: cruzar missing de renda com DebtRatio (células seguintes) — qualidade do campo, não só PD.

In [ ]:
# DebtRatio com vs sem renda
inc_na = df['MonthlyIncome'].isna()  # caso rode esta célula sozinha

print("DebtRatio mediana, renda OK:", df.loc[~inc_na, 'DebtRatio'].median())
print("DebtRatio mediana, renda NA:", df.loc[inc_na, 'DebtRatio'].median())

print("% DebtRatio > 1, renda OK:", (df.loc[~inc_na, 'DebtRatio'] > 1).mean())
print("% DebtRatio > 1, renda NA:", (df.loc[inc_na, 'DebtRatio'] > 1).mean())


Neste dataset, com renda ausente, o “ratio” muitas vezes não é dívida/renda — vira um número enorme.  
Se mediana e % > 1 disparam só no grupo “renda NA”, você prova que os outliers de DebtRatio da 1a/1b não são independentes do missing: é o mesmo problema. Fase 3: não dar cap cego no ratio sem olhar a flag de renda.


Renda OK: mediana DebtRatio = 0,30; só 6% com ratio > 1 → campo = fração dívida/renda.  
Renda NA: mediana = 1.159; 94% com “ratio” > 1 → não é ratio; provavelmente valor absoluto no lugar da fração.  

Os outliers extremos de DebtRatio (média 353, P90 ~1267) concentram-se em quem não tem MonthlyIncome.  


**Hipótese de missing (fechada o suficiente para a Fase 3):**

- PD e `DebtRatio` mudam quando a renda falta.
- **Não é “buraco = mais risco”:** quem não informa renda/dependentes tem **menos** default (PD 5,61% e 4,56% vs ~6,7%).

**Fase 3:** imputar renda **e** criar flag; não dropar 29k linhas; tratar `DebtRatio` alto **condicional** a renda missing.



## **3. Registros Duplicados**

In [ ]:
cols = [c for c in df.columns if c != 'Unnamed: 0']

n_id = df.duplicated().sum()
n_perfil = df.duplicated(subset=cols).sum()
n_perfil_com_primeira = df.duplicated(subset=cols, keep=False).sum()

print("Duplicados usando TODAS as colunas (inclui Unnamed: 0):", n_id)
print("Duplicados do PERFIL (sem Unnamed: 0), linhas extra:", n_perfil)
print("Linhas que fazem parte de algum grupo repetido:", n_perfil_com_primeira)

# Olhar os grupos (se n_perfil > 0)
if n_perfil > 0:
    dups = df[df.duplicated(subset=cols, keep=False)].sort_values(cols)
    display(dups.head(20))
    print("Grupos distintos:", dups.groupby(cols).ngroups)

Com Unnamed: 0: 0 duplicatas (o índice é único; esconde o resto).  
Sem Unnamed: 0: 609 linhas extra; 960 linhas em 351 perfis repetidos (~2,7 linhas/grupo).  
≈ 0,41% da base são cópias de perfil (features + target iguais; NaN conta como igual).  

Não é erro massivo de extração. Pode misturar coincidência (muitos zeros + renda NA) e cópias reais — ver os grupos no display.  


## **4. Distribuição do Target** 

In [ ]:
# 4. Distribuição do target (não altera df)
y = df['SeriousDlqin2yrs']

print("Contagem:\n", y.value_counts().sort_index())
print("\nPercentual:\n", y.value_counts(normalize=True).sort_index() * 100)
print("\nPD da base (média do target):", y.mean())
print("Razão maioria/minoria:", (y == 0).sum() / (y == 1).sum())

# Gráfico
ax = y.value_counts().sort_index().plot(kind='bar', rot=0, edgecolor='black')
ax.set_xticklabels(['0 = sem atraso 90+ em 2 anos', '1 = inadimplência grave'])
ax.set_ylabel('Número de clientes')
ax.set_title('SeriousDlqin2yrs — desbalanceamento')
plt.tight_layout()
plt.show()

SeriousDlqin2yrs:   
0 ≈ 93,32% (N ≈ 139.974);   
1 ≈ 6,68% (N ≈ 10.026).    
PD da carteira = 6,68%.   
Razão ≈ 14:1 (adimplente : inadimplente).  

Confirmado o desbalanceamento da Fase 1:  
- Acurácia sozinha engana (baseline burro ~93%).  
- Métricas: AUC-ROC + Recall (minimizar FN); KS na avaliação.  
- Fase 3: split estratificado no target.  
- Fase 4: lidar com desbalanceamento (pesos/threshold), não só “treinar e ver accuracy”.  


## **5. Análise Univariada**

Qual a forma da distribuição e onde estão os outliers?

In [ ]:
import matplotlib.pyplot as plt

def univariada_massa(s, nome, p=0.99, bins=40):
    """Histograma até o percentil p (só visual)."""
    x = s.dropna()
    corte = x.quantile(p)
    x = x[x <= corte]
    plt.figure(figsize=(7, 3.5))
    plt.hist(x, bins=bins, edgecolor='black')
    plt.title(f'{nome} — até P{int(p*100)} (n={len(x):,})')
    plt.xlabel(nome)
    plt.ylabel('Frequência')
    plt.tight_layout()
    plt.show()

def univariada_box(s, nome, p=0.99):
    x = s.dropna()
    x = x[x <= x.quantile(p)]
    plt.figure(figsize=(5, 3.5))
    plt.boxplot(x, vert=True)
    plt.title(f'{nome} — boxplot até P{int(p*100)}')
    plt.tight_layout()
    plt.show()

In [ ]:
# Forma “comportada”
univariada_massa(df['age'], 'age', p=1.0)          # p=1 = sem corte
univariada_box(df['age'], 'age', p=1.0)
univariada_massa(df['NumberOfOpenCreditLinesAndLoans'], 'linhas de crédito', p=1.0)

# Cauda (P99)
univariada_massa(df['RevolvingUtilizationOfUnsecuredLines'], 'utilização', p=0.99)
univariada_massa(df['MonthlyIncome'], 'MonthlyIncome', p=0.99)
univariada_massa(df['DebtRatio'], 'DebtRatio', p=0.99)

# Contagens: barras, não hist de 0–98
df['NumberOfTimes90DaysLate'].value_counts().sort_index().head(12).plot(kind='bar')
plt.title('90 dias late — valores mais comuns (sem 96/98 no foco)')
plt.show()

Gráficos confirmam a 1a (não mudam as decisões):
- age: concentrada ~41–63; idade 0 é ponto isolado.
- utilização / renda: histograma só é legível até P99; a massa é “normal”, o max não.
- DebtRatio na base toda é enganoso; univariada útil = só quem tem renda.
- atrasos: pico em 0; 96/98 não aparecem como cauda contínua.

Outliers da 1b continuam candidatos a cap/recodificar na Fase 3, não removidos aqui.

## **6a. Análise Bivariada - Visual**

A análise bivariada pergunta: a feature muda entre quem não defaultou (0) e quem defaultou (1)?

In [ ]:
def box_por_target(col, p=0.99, titulo=None):
    """Duas caixas: target 0 vs 1. Corta no P99 da coluna (só visual)."""
    s = df[col]
    corte = s.quantile(p)
    tmp = df.loc[s.notna() & (s <= corte), [col, 'SeriousDlqin2yrs']]
    tmp.boxplot(column=col, by='SeriousDlqin2yrs')
    plt.suptitle('')
    plt.title(titulo or f'{col} até P{int(p*100)} vs target')
    plt.xlabel('SeriousDlqin2yrs (0 = bom, 1 = default)')
    plt.tight_layout()
    plt.show()

In [ ]:
box_por_target('RevolvingUtilizationOfUnsecuredLines', p=0.99, titulo='Utilização vs default')
box_por_target('age', p=1.0, titulo='Idade vs default')  # sem corte
box_por_target('NumberOfTimes90DaysLate', p=0.99, titulo='Atrasos 90d vs default')  # 96/98 ainda podem poluir
box_por_target('MonthlyIncome', p=0.99, titulo='Renda vs default (só quem informou)')

In [ ]:
c = 'NumberOfTimes90DaysLate'
mask = df[c] < 96
tmp = df.loc[mask, [c, 'SeriousDlqin2yrs']]
tmp.boxplot(column=c, by='SeriousDlqin2yrs')
plt.title('Atrasos 90d vs target (sem 96/98)')
plt.suptitle('')
plt.show()

### **O que os boxplots mostram (corte visual no P99; atrasos 90d sem 96/98):**

- **Utilização:** a caixa do target 1 fica mais alta que a do 0 → inadimplente usa mais do limite.
- **Atrasos 90+:** no grupo 0 a massa é zero; no grupo 1 a caixa sobe (ainda com muitos zeros — um único atraso já importa; ver 6b).
- **Idade:** inadimplentes um pouco mais jovens; a separação é mais fraca que utilização/atraso.

Os gráficos **não** substituem a taxa de evento. A 6b quantifica PD.

| Variável / Plot | Leitura | Força |
|---|---|---|
| **Utilização** | Caixa do 1 mais alta que a do 0 | Forte |
| **Atraso 90d sem 96/98** | No 0, massa em zero; no 1, a caixa/cauda sobe (ainda há muitos zeros) | Forte |
| **Idade** | Default um pouco mais jovem | Média |
| **Renda (informada)** | Default um pouco mais baixa; muita sobreposição | Fraca / média |


## **6b. Análise Bivariada - Quantitativa** 

In [ ]:
# PD quando atraso = 0 vs ≥ 1 (melhor para 30/60/90)

def pd_zero_vs_positivo(col, excluir_96=True):
    s = df[col]
    if excluir_96:
        m = s < 96
    else:
        m = s.notna()
    y = df.loc[m, 'SeriousDlqin2yrs']
    x = s[m]
    print(col)
    print("  PD se == 0:  ", y[x == 0].mean(), " n=", (x == 0).sum())
    print("  PD se >= 1:  ", y[x >= 1].mean(), " n=", (x >= 1).sum())
    print()

for c in [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
]:
    pd_zero_vs_positivo(c)

In [ ]:
# PD por faixas (contínuas: utilização, idade, renda)

def pd_por_faixas(col, q=5, p99=True):
    """PD em quintis (Q1=mais baixo ... Q5=mais alto)."""
    s = df[col]
    y = df['SeriousDlqin2yrs']
    mask = s.notna()
    if p99:
        mask &= s <= s.quantile(0.99)
    x = s[mask]
    # qcut pode falhar se houver muitos empates; duplicates='drop'
    faixas = pd.qcut(x, q=q, duplicates='drop')
    tab = y[mask].groupby(faixas, observed=True).mean()
    print(col)
    print(tab)
    print()

pd_por_faixas('RevolvingUtilizationOfUnsecuredLines')
pd_por_faixas('age')
pd_por_faixas('MonthlyIncome')  # só quem tem renda (NaN já sai)

Base PD ≈ 6,7%.

Atrasos (sem 96/98): um único atraso já muda o risco.
- 30–59: 4,0% (n=126k) vs 20,4% (n=24k)
- 60–89: 5,1% vs 35,8% (n=7k)
- 90+: 4,6% vs 41,2% (n=8k)
Quanto mais grave a faixa, maior a PD no grupo ≥1.

Utilização (P99): PD 1,7–1,9% nos baixos até 18,5% no quintil ~0,68–1,09.
Idade: PD cai de 10,5% (≤39) para 2,4% (65–87).
Renda (informada): PD cai de 9,1% para 4,4% — sinal real, mais fraco.

Fase 4: priorizar atrasos + utilização; idade e renda como complemento.
Etapa 7: as 3 colunas de atraso provavelmente correlacionam entre si.

## **7. Correlação Entre Features**

In [ ]:
feat = [
    'RevolvingUtilizationOfUnsecuredLines', 'age',
    'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio',
    'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans',
    'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines',
    'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents',
]

# --- A) crua (NaNs de renda somem par a par) ---
corr_raw = df[feat].corr()

# --- B) para interpretação: 96/98 → NA; utilização e DebtRatio cap no P99 ---
tmp = df[feat].copy()
for c in ['NumberOfTime30-59DaysPastDueNotWorse',
          'NumberOfTime60-89DaysPastDueNotWorse',
          'NumberOfTimes90DaysLate']:
    tmp.loc[tmp[c] >= 96, c] = np.nan

for c in ['RevolvingUtilizationOfUnsecuredLines', 'DebtRatio']:
    tmp[c] = tmp[c].clip(upper=tmp[c].quantile(0.99))

corr = tmp.corr()

# pares |r| > 0.7
pares = (
    corr.where(np.triu(np.ones_like(corr, dtype=bool), k=1))
        .stack()
        .reset_index()
)
pares.columns = ['var1', 'var2', 'r']
print('Pares |r| > 0.7 (versão B):')
print(pares[pares['r'].abs() > 0.7].sort_values('r', key=abs, ascending=False))

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr,
    cmap='vlag',
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,       # escreve o r em cada célula
    fmt='.2f',        # duas casas (0.35, não 0.351234)
    annot_kws={'size': 8},
)
plt.title('Correlação (atrasos sem 96/98; util/DebtRatio cap P99)')
plt.tight_layout()
plt.show()

#### **7 — correlação (versão B: 96/98 → NA; utilização e DebtRatio cap P99)**

Pares com **|r| > 0,7: nenhum.** Depois de tirar o código 96/98, as três colunas de atraso **não** são a mesma variável.

- Podem **coexistir** em árvore/boosting.
- Em **logística**, ainda vale olhar VIF ou uma feature `total_atrasos` — mas **não** dropar duas delas só porque “atraso parece atrasar”.
- Correlação **entre features** ≠ poder preditivo (isso foi a 6b). `MonthlyIncome` com muitos NaN entra pouco no `corr` (pandas usa pares completos).

**Fase 3:** recodificar 96/98 **antes** de qualquer matriz de correlação / modelo. Não winsorizar 98 como se fosse atraso real.


## 8. Síntese e hipóteses para a Fase 3

**Em uma frase:** 150.000 linhas, ~6,7% de default (`SeriousDlqin2yrs = 1`).   
Métrica de sucesso = **AUC + Recall** (não acurácia).    
Na Fase 3: split **estratificado** no target.  
O que importa é achar o 7% que caloteia (Recall) e ordenar risco certo (AUC).

---

### O que prediz default (hipóteses confirmadas)

Base PD ≈ 6,7%. Um sinal já muda o risco:

| Sinal | Evidência (6b) | Força |
|--------|----------------|--------|
| Atraso 90+ dias (≥ 1) | PD 41% vs 4,6% | Alta |
| Atraso 60–89 (≥ 1) | PD 36% vs 5,1% | Alta |
| Atraso 30–59 (≥ 1) | PD 20% vs 4,0% | Alta |
| Utilização alta (quintil 5) | PD 18,5% vs ~2% nos baixos | Alta |
| Idade | 10,5% (mais jovens) → 2,4% (mais velhos) | Média |
| Renda (quando informada) | 9,1% → 4,4% | Média / fraca |

**Correlação (7), sem códigos 96/98 e com cap P99:** nenhum par com |r| > 0,7. As três colunas de atraso podem coexistir no boosting; em logística, só vigiar colinearidade (não dropar duas delas só pelo heatmap).

**Hipótese ainda aberta (baixa prioridade):** linhas de crédito / imóvel — sinal mais fraco na 6b; não ranquear no mesmo nível que atrasos e utilização.

---

### O que está quebrado → o que fazer na Fase 3

Fazer **só no notebook 03**. Aqui não se altera o `df`.

| # | Problema | Evidência | Ação |
|---|----------|-----------|------|
| 1 | `Unnamed: 0` | Índice, não risco | Drop; não usar como feature |
| 2 | Renda `NaN` (~20%, 29.731) | PD 5,61% (NA) vs 6,95% (ok) — **não é MCAR** | **Não dropar.** Imputar + flag `income_was_missing`. Incluir renda = 0 (1.634) no mesmo pacote |
| 3 | `DebtRatio` extremo | Mediana 0,30 se renda ok (6% > 1) vs 1.159 se renda NA (94% > 1) | Tratar **condicional** à flag: com renda = ratio; sem renda ≠ ratio (não dar cap cego na base toda) |
| 4 | Dependentes `NaN` (~2,6%) | PD 4,56% vs 6,74% | Imputar (moda/0) + flag se for informativo |
| 5 | Atrasos = 96 ou 98 | Código especial, não 98 eventos reais | Recodificar (NaN, cap ou dummy) **antes** do modelo |
| 6 | Utilização até 50.708 | Massa 0–1 (P99 ≈ 1,09); cauda quebra a média | Cap P99 e/ou flag `> 10` |
| 7 | `age == 0` | 1 linha (impossível) | Remover ou imputar |
| 8 | Perfis duplicados (sem ID) | 609 linhas extra (~0,4%) | Opcional `drop_duplicates(keep='first')`. Prioridade baixa |

---

### Ordem sugerida no notebook 03

1. Drop `Unnamed: 0`
2. Recodificar 96/98
3. Flags de missing + imputação — **mediana/P99 só no treino** (evitar leakage)
4. `DebtRatio` separado por flag de renda
5. Cap de utilização (e renda extrema, se fizer)
6. Tratar `age == 0`
7. Duplicados (opcional)
8. `train_test_split(..., stratify=y)` — depois das regras de recodificação; estatísticas de imputação/cap **só no treino**

---

### Não fazer

- Dropar as 29 mil linhas sem renda (sai o grupo **melhor**).
- Usar média de utilização, `DebtRatio` ou atrasos como “cliente típico”.
- Usar acurácia ou limiar 0,5 como critério de sucesso (Fases 4–5).